In [ ]:
import os
import re
import unicodedata
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")
GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

VALIDATION_IDS_FILE = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

CHUNK_SIZE = 100_000

PAIR_THRESHOLDS = [2, 5, 10, 20, 50, 100]
MAX_PAIR_FREQUENCY = max(PAIR_THRESHOLDS)
TARGET_PAIR_POSTING_MAX_FREQUENCY = 500

print("Thresholds:", PAIR_THRESHOLDS)

In [ ]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return set()

    return {
        token
        for token in value.split()
        if len(token) >= 2
    }

In [ ]:
with open(VALIDATION_IDS_FILE, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

validation_s1_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    if len(selected) > 0:
        validation_s1_parts.append(selected)

validation_s1 = pd.concat(
    validation_s1_parts,
    ignore_index=True
)

validation_s1 = validation_s1.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

validation_s1 = validation_s1.reset_index(drop=True)

print("Validation S1 rows:", len(validation_s1))

In [ ]:
s1_id_to_index = {
    s1_id: i
    for i, s1_id in enumerate(
        validation_s1["s1_id"]
    )
}

print(
    "S1 index entries:",
    len(s1_id_to_index)
)

In [ ]:
token_to_id = {}
next_token_id = 1

name_pair_frequency = Counter()
address_pair_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    # -----------------------
    # NAME
    # -----------------------
    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    name_token_ids = []

    for token in name_tokens:

        if token not in token_to_id:
            token_to_id[token] = next_token_id
            next_token_id += 1

        name_token_ids.append(
            token_to_id[token]
        )

    if len(name_token_ids) >= 2:
        name_pair_frequency.update(
            combinations(name_token_ids, 2)
        )

    # -----------------------
    # ADDRESS
    # -----------------------
    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    address_token_ids = []

    for token in address_tokens:

        if token not in token_to_id:
            token_to_id[token] = next_token_id
            next_token_id += 1

        address_token_ids.append(
            token_to_id[token]
        )

    if len(address_token_ids) >= 2:
        address_pair_frequency.update(
            combinations(address_token_ids, 2)
        )

print("Unique tokens:", len(token_to_id))
print("Unique name pairs:", len(name_pair_frequency))
print("Unique address pairs:", len(address_pair_frequency))

In [ ]:
def frequency_summary(counter):
    values = np.array(
        list(counter.values()),
        dtype=np.int64
    )

    return pd.Series(
        values
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )


print("NAME PAIR FREQUENCY")
display(
    frequency_summary(name_pair_frequency)
)

print("\nADDRESS PAIR FREQUENCY")
display(
    frequency_summary(address_pair_frequency)
)

In [ ]:
name_pair_to_s1 = defaultdict(list)
address_pair_to_s1 = defaultdict(list)

for s1_idx, row in enumerate(
    validation_s1.itertuples(index=False)
):

    # -----------------------
    # NAME PAIRS
    # -----------------------
    name_token_ids = []

    for token in sorted(
        get_tokens(row.s1_name)
    ):
        name_token_ids.append(
            token_to_id[token]
        )

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            if (
                name_pair_frequency.get(pair, 0)
                <= TARGET_PAIR_POSTING_MAX_FREQUENCY
            ):
                name_pair_to_s1[pair].append(
                    s1_idx
                )

    # -----------------------
    # ADDRESS PAIRS
    # -----------------------
    address_token_ids = []

    for token in sorted(
        get_tokens(row.s1_address)
    ):
        address_token_ids.append(
            token_to_id[token]
        )

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            if (
                address_pair_frequency.get(pair, 0)
                <= TARGET_PAIR_POSTING_MAX_FREQUENCY
            ):
                address_pair_to_s1[pair].append(
                    s1_idx
                )

print(
    "Indexed name pairs:",
    len(name_pair_to_s1)
)

print(
    "Indexed address pairs:",
    len(address_pair_to_s1)
)

In [ ]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):

    selected = chunk[
        chunk["source1_entity_id"].isin(
            validation_s1_ids
        )
    ]

    for row in selected.itertuples(index=False):

        value = row.matched_entity_ids.strip()

        if value:
            matches = {
                x.strip()
                for x in value.split(",")
                if x.strip()
            }
        else:
            matches = set()

        validation_ground_truth[
            row.source1_entity_id
        ] = matches


true_target_to_s1_idx = {}

for s1_id, target_ids in validation_ground_truth.items():

    s1_idx = s1_id_to_index[s1_id]

    for target_id in target_ids:

        true_target_to_s1_idx[target_id] = s1_idx


total_true_matches = len(
    true_target_to_s1_idx
)

print(
    "Validation true matches:",
    total_true_matches
)

In [ ]:
def evaluate_pair_blocking_source(
    source_path,
    source_name
):
    candidate_pairs = {
        threshold: 0
        for threshold in PAIR_THRESHOLDS
    }

    candidate_counts_per_s1 = {
        threshold: np.zeros(
            len(validation_s1),
            dtype=np.int64
        )
        for threshold in PAIR_THRESHOLDS
    }

    true_hits = {
        threshold: set()
        for threshold in PAIR_THRESHOLDS
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            target_id = row.entity_id

            # -----------------------
            # Convert target tokens to IDs
            # -----------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            # Build only the pairs that actually
            # exist in the validation S1 vocabulary.
            name_pairs = set(
                combinations(
                    sorted(name_token_ids),
                    2
                )
            )

            address_pairs = set(
                combinations(
                    sorted(address_token_ids),
                    2
                )
            )

            for threshold in PAIR_THRESHOLDS:

                candidate_s1 = set()

                # -----------------------
                # Name pair candidates
                # -----------------------
                for pair in name_pairs:

                    frequency = (
                        name_pair_frequency.get(
                            pair,
                            0
                        )
                    )

                    if (
                        0 < frequency <= threshold
                    ):
                        candidate_s1.update(
                            name_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                # -----------------------
                # Address pair candidates
                # -----------------------
                for pair in address_pairs:

                    frequency = (
                        address_pair_frequency.get(
                            pair,
                            0
                        )
                    )

                    if (
                        0 < frequency <= threshold
                    ):
                        candidate_s1.update(
                            address_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                count = len(candidate_s1)

                candidate_pairs[
                    threshold
                ] += count

                # Candidate load for each S1.
                for s1_idx in candidate_s1:
                    candidate_counts_per_s1[
                        threshold
                    ][s1_idx] += 1

                # True match recall
                if target_id in true_target_to_s1_idx:

                    true_s1_idx = (
                        true_target_to_s1_idx[
                            target_id
                        ]
                    )

                    if true_s1_idx in candidate_s1:

                        true_hits[
                            threshold
                        ].add(target_id)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    return {
        "candidate_pairs": candidate_pairs,
        "candidate_counts_per_s1": candidate_counts_per_s1,
        "true_hits": true_hits
    }

In [ ]:
print("Starting S2...")

pair_results_s2 = evaluate_pair_blocking_source(
    SOURCE2,
    "S2"
)

print("S2 complete.")

In [ ]:
print("Starting S3...")

pair_results_s3 = evaluate_pair_blocking_source(
    SOURCE3,
    "S3"
)

print("S3 complete.")

In [ ]:
pair_blocking_results = []

for threshold in PAIR_THRESHOLDS:

    true_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    total_candidates = (
        pair_results_s2["candidate_pairs"][threshold]
        +
        pair_results_s3["candidate_pairs"][threshold]
    )

    candidates_per_s1 = (
        pair_results_s2[
            "candidate_counts_per_s1"
        ][threshold]
        +
        pair_results_s3[
            "candidate_counts_per_s1"
        ][threshold]
    )

    pair_blocking_results.append({

        "pair_frequency_threshold": threshold,

        "candidate_pairs": total_candidates,

        "true_matches_captured": len(
            true_hits
        ),

        "total_true_matches": total_true_matches,

        "candidate_recall": (
            len(true_hits)
            / total_true_matches
        ),

        "avg_candidates_per_s1": (
            total_candidates
            / len(validation_s1)
        ),

        "median_candidates_per_s1": (
            np.median(candidates_per_s1)
        ),

        "p95_candidates_per_s1": (
            np.percentile(
                candidates_per_s1,
                95
            )
        ),

        "max_candidates_per_s1": (
            np.max(candidates_per_s1)
        )
    })


pair_blocking_results_df = pd.DataFrame(
    pair_blocking_results
)

pair_blocking_results_df

In [ ]:
pair_blocking_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "blocking_experiment_02_pair_results.csv"
    ),
    index=False
)

print("Saved pair-blocking results.")

In [ ]:
import pickle
import os

CAPTURED_PATH = os.path.join(
    OUTPUT_DIR,
    "blocking_threshold_10_captured.pkl"
)

with open(CAPTURED_PATH, "rb") as f:
    current_captured = pickle.load(f)

print(
    "Current blocker captured:",
    len(current_captured)
)

pair_results_comparison = []

for threshold in PAIR_THRESHOLDS:

    pair_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    overlap = (
        pair_hits
        &
        current_captured
    )

    new_from_pair = (
        pair_hits
        -
        current_captured
    )

    union_hits = (
        pair_hits
        |
        current_captured
    )

    pair_results_comparison.append({
        "pair_threshold": threshold,
        "pair_hits": len(pair_hits),
        "overlap_with_current": len(overlap),
        "new_matches_from_pair": len(new_from_pair),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

pair_comparison_df = pd.DataFrame(
    pair_results_comparison
)

pair_comparison_df

In [ ]:
pair_results_comparison = []

for threshold in PAIR_THRESHOLDS:

    pair_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    overlap = (
        pair_hits
        &
        current_captured
    )

    new_from_pair = (
        pair_hits
        -
        current_captured
    )

    union_hits = (
        pair_hits
        |
        current_captured
    )

    pair_results_comparison.append({
        "pair_threshold": threshold,
        "pair_hits": len(pair_hits),
        "overlap_with_current": len(overlap),
        "new_matches_from_pair": len(new_from_pair),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

pair_comparison_df = pd.DataFrame(
    pair_results_comparison
)

pair_comparison_df

In [ ]:
total_true_matches = len(true_target_to_s1_idx)

print(
    "Total validation true matches:",
    total_true_matches
)

In [ ]:
import pickle

pair_baseline_state = {
    "pair_true_hits": {
        threshold: (
            pair_results_s2["true_hits"][threshold]
            |
            pair_results_s3["true_hits"][threshold]
        )
        for threshold in PAIR_THRESHOLDS
    },
    "total_true_matches": total_true_matches
}

pair_state_path = os.path.join(
    OUTPUT_DIR,
    "blocking_experiment_02_pair_state.pkl"
)

with open(pair_state_path, "wb") as f:
    pickle.dump(
        pair_baseline_state,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", pair_state_path)

In [ ]:
UNION_THRESHOLDS = [2, 5, 10]

In [ ]:
def evaluate_union_candidate_volume(
    source_path,
    source_name,
    thresholds=UNION_THRESHOLDS
):
    total_candidates = {
        threshold: 0
        for threshold in thresholds
    }

    candidate_counts_per_s1 = {
        threshold: np.zeros(
            len(validation_s1),
            dtype=np.int32
        )
        for threshold in thresholds
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            target_name_norm = normalize_text(
                row.business_name
            )

            target_address_norm = normalize_text(
                row.business_address
            )

            # -------------------------------------------------
            # Existing current blocker
            # exact normalized name
            # + name token <=10
            # + address token <=10
            # -------------------------------------------------
            current_candidates = set()

            if target_name_norm:
                current_candidates.update(
                    name_exact_map.get(
                        target_name_norm,
                        []
                    )
                )

            for token in get_tokens(row.business_name):

                freq = name_token_frequency.get(
                    token,
                    0
                )

                if 0 < freq <= 10:
                    current_candidates.update(
                        name_token_to_s1.get(
                            token,
                            []
                        )
                    )

            for token in get_tokens(row.business_address):

                freq = address_token_frequency.get(
                    token,
                    0
                )

                if 0 < freq <= 10:
                    current_candidates.update(
                        address_token_to_s1.get(
                            token,
                            []
                        )
                    )

            # -------------------------------------------------
            # Pair candidates
            # -------------------------------------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            name_pairs = set(
                combinations(
                    sorted(name_token_ids),
                    2
                )
            )

            address_pairs = set(
                combinations(
                    sorted(address_token_ids),
                    2
                )
            )

            for threshold in thresholds:

                union_candidates = set(
                    current_candidates
                )

                # Name pair candidates
                for pair in name_pairs:

                    freq = name_pair_frequency.get(
                        pair,
                        0
                    )

                    if 0 < freq <= threshold:
                        union_candidates.update(
                            name_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                # Address pair candidates
                for pair in address_pairs:

                    freq = address_pair_frequency.get(
                        pair,
                        0
                    )

                    if 0 < freq <= threshold:
                        union_candidates.update(
                            address_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                count = len(union_candidates)

                total_candidates[
                    threshold
                ] += count

                for s1_idx in union_candidates:
                    candidate_counts_per_s1[
                        threshold
                    ][s1_idx] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    return {
        "total_candidates": total_candidates,
        "candidate_counts_per_s1": candidate_counts_per_s1
    }

In [ ]:
# ---------------------------------------------------------
# Rebuild token-frequency maps
# ---------------------------------------------------------

from collections import Counter, defaultdict

name_token_frequency = Counter()
address_token_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    name_token_frequency.update(
        get_tokens(row.s1_name)
    )

    address_token_frequency.update(
        get_tokens(row.s1_address)
    )
# ---------------------------------------------------------
# Rebuild ALL candidate indexes using integer S1 indices
# ---------------------------------------------------------

name_exact_map = defaultdict(list)

name_token_to_s1 = defaultdict(list)
address_token_to_s1 = defaultdict(list)

name_pair_to_s1 = defaultdict(list)
address_pair_to_s1 = defaultdict(list)


for s1_idx, row in enumerate(
    validation_s1.itertuples(index=False)
):

    # =====================================================
    # NAME
    # =====================================================

    name_norm = normalize_text(
        row.s1_name
    )

    if name_norm:
        name_exact_map[
            name_norm
        ].append(s1_idx)

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

for token in name_tokens:

    frequency = name_token_frequency.get(
        token,
        0
    )

    if 0 < frequency <= 10:
        name_token_to_s1[token].append(
            s1_idx
        )

    # Name token pairs
    if len(name_tokens) >= 2:

        name_ids = [
            token_to_id[token]
            for token in name_tokens
            if token in token_to_id
        ]

        for pair in combinations(
            sorted(name_ids),
            2
        ):

            frequency = name_pair_frequency.get(
                pair,
                0
            )

            if (
                0 < frequency <= TARGET_PAIR_POSTING_MAX_FREQUENCY
            ):
                name_pair_to_s1[
                    pair
                ].append(s1_idx)


    # =====================================================
    # ADDRESS
    # =====================================================

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    for token in address_tokens:

        # Single-token blocking
        if token in address_token_frequency:
            address_token_to_s1[
                token
            ].append(s1_idx)

    # Address token pairs
    if len(address_tokens) >= 2:

        address_ids = [
            token_to_id[token]
            for token in address_tokens
            if token in token_to_id
        ]

        for pair in combinations(
            sorted(address_ids),
            2
        ):

            frequency = address_pair_frequency.get(
                pair,
                0
            )

            if (
                0 < frequency <= TARGET_PAIR_POSTING_MAX_FREQUENCY
            ):
                address_pair_to_s1[
                    pair
                ].append(s1_idx)


print("Exact-name keys:", len(name_exact_map))
print("Name-token keys:", len(name_token_to_s1))
print("Address-token keys:", len(address_token_to_s1))
print("Name-pair keys:", len(name_pair_to_s1))
print("Address-pair keys:", len(address_pair_to_s1))

In [ ]:
maps_to_check = {
    "name_exact_map": name_exact_map,
    "name_token_to_s1": name_token_to_s1,
    "address_token_to_s1": address_token_to_s1,
    "name_pair_to_s1": name_pair_to_s1,
    "address_pair_to_s1": address_pair_to_s1,
}

for map_name, mapping in maps_to_check.items():

    sample_values = []

    for values in mapping.values():
        if values:
            sample_values.extend(values[:3])

        if len(sample_values) >= 3:
            break

    print(
        f"{map_name:25} ->",
        [type(x).__name__ for x in sample_values],
        sample_values
    )

In [ ]:
sample_key = next(iter(name_exact_map))

print(
    "Sample key:",
    sample_key
)

print(
    "Sample mapped values:",
    name_exact_map[sample_key][:5]
)

print(
    "Types:",
    [type(x).__name__ for x in name_exact_map[sample_key][:5]]
)

In [ ]:
union_s2 = evaluate_union_candidate_volume(
    SOURCE2,
    "S2"
)


In [ ]:
union_s3 = evaluate_union_candidate_volume(
    SOURCE3,
    "S3"
)

In [ ]:
union_volume_results = []

for threshold in UNION_THRESHOLDS:

    total_candidates = (
        union_s2["total_candidates"][threshold]
        +
        union_s3["total_candidates"][threshold]
    )

    candidates_per_s1 = (
        union_s2["candidate_counts_per_s1"][threshold]
        +
        union_s3["candidate_counts_per_s1"][threshold]
    )

    union_volume_results.append({
        "threshold": threshold,
        "candidate_pairs": total_candidates,
        "avg_candidates_per_s1": (
            total_candidates / len(validation_s1)
        ),
        "median_candidates_per_s1": np.median(
            candidates_per_s1
        ),
        "p95_candidates_per_s1": np.percentile(
            candidates_per_s1, 95
        ),
        "max_candidates_per_s1": np.max(
            candidates_per_s1
        )
    })

union_volume_df = pd.DataFrame(
    union_volume_results
)

union_volume_df

In [ ]:
final_blocking_tradeoff = (
    union_volume_df
    .merge(
        pair_comparison_df[
            [
                "pair_threshold",
                "union_true_matches",
                "union_recall"
            ]
        ],
        left_on="threshold",
        right_on="pair_threshold",
        how="left"
    )
    .drop(columns=["pair_threshold"])
)

final_blocking_tradeoff

In [ ]:
candidate_tail = []

# Use threshold 2
counts_s2 = union_s2["candidate_counts_per_s1"][2]
counts_s3 = union_s3["candidate_counts_per_s1"][2]

combined_counts = counts_s2 + counts_s3

for threshold in [1000, 5000, 10000, 50000, 100000]:
    candidate_tail.append({
        "candidate_threshold": threshold,
        "s1_entities_above_threshold": int(
            (combined_counts > threshold).sum()
        ),
        "percentage_of_validation_s1": (
            (combined_counts > threshold).mean() * 100
        )
    })

candidate_tail_df = pd.DataFrame(candidate_tail)

candidate_tail_df

In [ ]:
pair_hits_threshold_2 = (
    pair_results_s2["true_hits"][2]
    |
    pair_results_s3["true_hits"][2]
)

current_blocker_hits_threshold_2 = (
    pair_hits_threshold_2
    |
    current_captured
)

all_true_targets = set(
    true_target_to_s1_idx.keys()
)

missed_after_threshold_2 = (
    all_true_targets
    -
    current_blocker_hits_threshold_2
)

print(
    "Total true matches:",
    len(all_true_targets)
)

print(
    "Captured:",
    len(current_blocker_hits_threshold_2)
)

print(
    "Still missed:",
    len(missed_after_threshold_2)
)

print(
    "Recall:",
    len(current_blocker_hits_threshold_2)
    / len(all_true_targets)
)

In [ ]:
import random

rng = random.Random(42)

sample_size = min(
    1000,
    len(missed_after_threshold_2)
)

sampled_remaining_misses = set(
    rng.sample(
        list(missed_after_threshold_2),
        sample_size
    )
)

print(
    "Sampled remaining misses:",
    len(sampled_remaining_misses)
)

In [ ]:
threshold_2_counts = (
    union_s2["candidate_counts_per_s1"][2]
    +
    union_s3["candidate_counts_per_s1"][2]
)

heavy_s1 = pd.DataFrame({
    "s1_index": np.arange(len(threshold_2_counts)),
    "candidate_count": threshold_2_counts
})

heavy_s1 = (
    heavy_s1
    .sort_values(
        "candidate_count",
        ascending=False
    )
    .head(10)
)

heavy_s1 = heavy_s1.merge(
    validation_s1[
        [
            "s1_id",
            "s1_name",
            "s1_address",
            "s1_country"
        ]
    ].reset_index().rename(
        columns={"index": "s1_index"}
    ),
    on="s1_index",
    how="left"
)

heavy_s1

In [ ]:
def token_frequency_details(value, frequency_map):
    tokens = get_tokens(value)

    return sorted(
        [
            (token, frequency_map.get(token, 0))
            for token in tokens
        ],
        key=lambda x: x[1],
        reverse=True
    )


heavy_s1["name_token_frequencies"] = (
    heavy_s1["s1_name"]
    .map(
        lambda x: token_frequency_details(
            x,
            name_token_frequency
        )
    )
)

heavy_s1["address_token_frequencies"] = (
    heavy_s1["s1_address"]
    .map(
        lambda x: token_frequency_details(
            x,
            address_token_frequency
        )
    )
)

heavy_s1[
    [
        "s1_id",
        "candidate_count",
        "s1_name",
        "s1_address",
        "s1_country",
        "name_token_frequencies",
        "address_token_frequencies"
    ]
].head(30)

In [ ]:
for cutoff in [1000, 5000, 10000, 50000, 100000]:
    total_from_heavy = (
        threshold_2_counts[
            threshold_2_counts > cutoff
        ].sum()
    )

    print(
        f"> {cutoff:6,d} candidates: "
        f"{(threshold_2_counts > cutoff).sum():6,d} S1s | "
        f"{total_from_heavy:,} candidate assignments"
    )

In [ ]:
heavy_indices = set(
    heavy_s1.head(10)["s1_index"].astype(int)
)

heavy_records = validation_s1.iloc[
    list(heavy_indices)
]

heavy_name_tokens = set()
heavy_address_tokens = set()

for row in heavy_records.itertuples(index=False):
    heavy_name_tokens.update(
        get_tokens(row.s1_name)
    )
    heavy_address_tokens.update(
        get_tokens(row.s1_address)
    )

print(
    "Heavy-S1 name tokens:",
    len(heavy_name_tokens)
)

print(
    "Heavy-S1 address tokens:",
    len(heavy_address_tokens)
)

In [ ]:
target_name_frequency = Counter()
target_address_frequency = Counter()

def scan_target_token_frequency(
    source_path,
    source_name
):
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            name_tokens = (
                get_tokens(row.business_name)
                &
                heavy_name_tokens
            )

            address_tokens = (
                get_tokens(row.business_address)
                &
                heavy_address_tokens
            )

            target_name_frequency.update(
                name_tokens
            )

            target_address_frequency.update(
                address_tokens
            )

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

In [ ]:
scan_target_token_frequency(
    SOURCE2,
    "S2"
)

scan_target_token_frequency(
    SOURCE3,
    "S3"
)

In [69]:
heavy_token_comparison = []

for token in sorted(
    heavy_name_tokens
):
    heavy_token_comparison.append({
        "field": "name",
        "token": token,
        "validation_s1_frequency": (
            name_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_name_frequency.get(
                token,
                0
            )
        )
    })

for token in sorted(
    heavy_address_tokens
):
    heavy_token_comparison.append({
        "field": "address",
        "token": token,
        "validation_s1_frequency": (
            address_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_address_frequency.get(
                token,
                0
            )
        )
    })

heavy_token_comparison_df = pd.DataFrame(
    heavy_token_comparison
)

heavy_token_comparison_df = (
    heavy_token_comparison_df
    .sort_values(
        "s2_s3_target_frequency",
        ascending=False
    )
)

heavy_token_comparison_df.head(100)

,field,token,validation_s1_frequency,s2_s3_target_frequency
41,address,road,45852,1288416
44,address,street,31080,694453
21,address,drive,22288,457748
0,name,center,3455,414505
16,address,avenue,19106,405843
48,address,tx,13179,307571
27,address,lane,10699,243506
35,address,ny,10239,237129
49,address,unit,19020,219006
37,address,park,5159,201478


In [ ]:
heavy_s1[
    [
        "s1_id",
        "candidate_count",
        "s1_name",
        "s1_address",
        "name_token_frequencies",
        "address_token_frequencies"
    ]
].head(10)

In [ ]:
heavy_pair_keys = set()

for row in heavy_records.itertuples(index=False):

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    if len(name_tokens) >= 2:
        heavy_pair_keys.update(
            combinations(name_tokens, 2)
        )

    if len(address_tokens) >= 2:
        heavy_pair_keys.update(
            combinations(address_tokens, 2)
        )

print("Heavy S1 token pairs:", len(heavy_pair_keys))

In [ ]:
target_pair_frequency = Counter()

def scan_target_pair_frequency(source_path, source_name):

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            name_tokens = get_tokens(
                row.business_name
            )

            address_tokens = get_tokens(
                row.business_address
            )

            name_pairs = combinations(
                sorted(name_tokens),
                2
            )

            address_pairs = combinations(
                sorted(address_tokens),
                2
            )

            for pair in name_pairs:

                if pair in heavy_pair_keys:
                    target_pair_frequency[pair] += 1

            for pair in address_pairs:

                if pair in heavy_pair_keys:
                    target_pair_frequency[pair] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

In [ ]:
scan_target_pair_frequency(
    SOURCE2,
    "S2"
)

scan_target_pair_frequency(
    SOURCE3,
    "S3"
)

In [ ]:
# Build STRING-based pair frequencies specifically for the
# heavy S1 records so the keys match heavy_pair_keys.

heavy_name_pair_frequency = Counter()
heavy_address_pair_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    if len(name_tokens) >= 2:
        heavy_name_pair_frequency.update(
            combinations(name_tokens, 2)
        )

    if len(address_tokens) >= 2:
        heavy_address_pair_frequency.update(
            combinations(address_tokens, 2)
        )


heavy_pair_comparison = []

for pair in heavy_pair_keys:

    name_freq = heavy_name_pair_frequency.get(
        pair,
        0
    )

    address_freq = heavy_address_pair_frequency.get(
        pair,
        0
    )

    # Keep the two fields separate.
    heavy_pair_comparison.append({
        "token_1": pair[0],
        "token_2": pair[1],
        "name_s1_frequency": name_freq,
        "address_s1_frequency": address_freq,
        "target_frequency": (
            target_pair_frequency.get(pair, 0)
        )
    })


heavy_pair_comparison_df = (
    pd.DataFrame(heavy_pair_comparison)
    .sort_values(
        "target_frequency",
        ascending=False
    )
)

heavy_pair_comparison_df.head(100)

In [ ]:
heavy_pair_comparison_df[
    [
        "token_1",
        "token_2",
        "name_s1_frequency",
        "address_s1_frequency",
        "target_frequency"
    ]
].head(50)

In [ ]:
PAIR_S1_THRESHOLD = 2

active_name_pairs = {
    pair
    for pair, freq in name_pair_frequency.items()
    if 0 < freq <= PAIR_S1_THRESHOLD
}

active_address_pairs = {
    pair
    for pair, freq in address_pair_frequency.items()
    if 0 < freq <= PAIR_S1_THRESHOLD
}

print("Active name pairs:", len(active_name_pairs))
print("Active address pairs:", len(active_address_pairs))

In [ ]:
target_name_pair_frequency_2 = Counter()
target_address_pair_frequency_2 = Counter()


def scan_active_pair_frequency(
    source_path,
    source_name
):
    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            if len(name_token_ids) >= 2:

                for pair in combinations(
                    sorted(name_token_ids),
                    2
                ):
                    if pair in active_name_pairs:
                        target_name_pair_frequency_2[pair] += 1

            if len(address_token_ids) >= 2:

                for pair in combinations(
                    sorted(address_token_ids),
                    2
                ):
                    if pair in active_address_pairs:
                        target_address_pair_frequency_2[pair] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

In [ ]:
scan_active_pair_frequency(
    SOURCE2,
    "S2"
)

scan_active_pair_frequency(
    SOURCE3,
    "S3"
)

In [ ]:
target_pair_frequency_2 = Counter()

target_pair_frequency_2.update(
    target_name_pair_frequency_2
)

target_pair_frequency_2.update(
    target_address_pair_frequency_2
)

print(
    "Active pairs with target occurrences:",
    len(target_pair_frequency_2)
)

print(
    target_pair_frequency_2.most_common(30)
)

In [ ]:
# ---------------------------------------------------------
# Rebuild the true-target lookup
# ---------------------------------------------------------

true_target_to_s1_idx = {}

for s1_id, target_ids in validation_ground_truth.items():
    s1_idx = s1_id_to_index[s1_id]

    for target_id in target_ids:
        true_target_to_s1_idx[target_id] = s1_idx

print(
    "Unique validation true targets:",
    len(true_target_to_s1_idx)
)

In [ ]:
true_target_min_pair_freq = {
    target_id: np.inf
    for target_id in true_target_to_s1_idx
}

print(
    "Targets to evaluate:",
    len(true_target_min_pair_freq)
)

In [ ]:
def evaluate_true_target_pair_selectivity(
    source_path,
    source_name
):
    rows_scanned = 0
    true_targets_seen = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        rows_in_chunk = len(chunk)
        rows_scanned += rows_in_chunk

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id

            if target_id not in true_target_min_pair_freq:
                continue

            true_targets_seen += 1

            min_freq = true_target_min_pair_freq[
                target_id
            ]

            # -----------------------------
            # NAME PAIRS
            # -----------------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            if len(name_token_ids) >= 2:

                for pair in combinations(
                    sorted(name_token_ids),
                    2
                ):

                    if pair not in active_name_pairs:
                        continue

                    freq = target_name_pair_frequency_2.get(
                        pair,
                        np.inf
                    )

                    if freq < min_freq:
                        min_freq = freq

            # -----------------------------
            # ADDRESS PAIRS
            # -----------------------------
            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            if len(address_token_ids) >= 2:

                for pair in combinations(
                    sorted(address_token_ids),
                    2
                ):

                    if pair not in active_address_pairs:
                        continue

                    freq = target_address_pair_frequency_2.get(
                        pair,
                        np.inf
                    )

                    if freq < min_freq:
                        min_freq = freq

            true_target_min_pair_freq[
                target_id
            ] = min_freq

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{rows_scanned:,} rows scanned | "
                f"{true_targets_seen:,} true targets seen"
            )

    print(
        f"{source_name} COMPLETE: "
        f"{rows_scanned:,} rows scanned | "
        f"{true_targets_seen:,} true targets seen"
    )

In [ ]:
evaluate_true_target_pair_selectivity(
    SOURCE2,
    "S2"
)

evaluate_true_target_pair_selectivity(
    SOURCE3,
    "S3"
)

In [ ]:
target_selectivity_results = []

for threshold in TARGET_PAIR_THRESHOLDS:

    surviving = sum(
        freq <= threshold
        for freq in true_target_min_pair_freq.values()
    )

    target_selectivity_results.append({
        "target_pair_frequency_threshold": threshold,
        "true_matches_surviving": surviving,
        "total_true_matches": total_true_matches,
        "candidate_recall": (
            surviving / total_true_matches
        )
    })

target_selectivity_df = pd.DataFrame(
    target_selectivity_results
)

target_selectivity_df

In [ ]:
import numpy as np
import pandas as pd

# Load / rebuild current blocker true targets if needed
# This should represent the targets captured by:
# exact name + name token <=10 + address token <=10 + pair <=2

current_captured_true_targets = set()

# If you already have this set saved, load/use it here.
# Example:
# current_captured_true_targets = captured_true_target_ids

thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_targets = len(true_target_to_s1_idx)

rows = []

for threshold in thresholds:
    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    newly_recovered = target_route_targets - current_captured_true_targets
    union_targets = current_captured_true_targets | target_route_targets

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "new_true_matches_added": len(newly_recovered),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_targets
    })

target_union_df = pd.DataFrame(rows)
target_union_df

In [ ]:
import pickle
import os

path = "/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/blocking_threshold_10_captured.pkl"

with open(path, "rb") as f:
    obj = pickle.load(f)

print("Object type:", type(obj))

if isinstance(obj, dict):
    print("Dictionary keys:")
    for k, v in obj.items():
        print(f"  {k}: type={type(v)}, size={len(v) if hasattr(v, '__len__') else 'NA'}")
else:
    print("Size:", len(obj) if hasattr(obj, "__len__") else "NA")

In [ ]:
def find_target_id_set(obj):
    """
    Find a set/list containing S2-/S3- target IDs.
    """
    if isinstance(obj, (set, list, tuple)):
        sample = list(obj)[:20]
        if any(
            isinstance(x, str) and (x.startswith("S2-") or x.startswith("S3-"))
            for x in sample
        ):
            return set(obj)

    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (set, list, tuple)):
                sample = list(value)[:20]

                if any(
                    isinstance(x, str)
                    and (x.startswith("S2-") or x.startswith("S3-"))
                    for x in sample
                ):
                    print("Using key:", key)
                    return set(value)

    raise ValueError("Could not find S2/S3 target ID set in pickle.")


current_captured_true_targets = find_target_id_set(obj)

print(
    "Current blocker captured true targets:",
    len(current_captured_true_targets)
)

In [ ]:
thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_targets = len(true_target_to_s1_idx)

rows = []

for threshold in thresholds:

    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    newly_recovered = (
        target_route_targets
        - current_captured_true_targets
    )

    union_targets = (
        current_captured_true_targets
        | target_route_targets
    )

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "current_blocker_true_matches": len(current_captured_true_targets),
        "new_true_matches_added": len(newly_recovered),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_targets
    })

target_union_df = pd.DataFrame(rows)

target_union_df

In [ ]:
from pathlib import Path
import pickle

filename = "blocking_experiment_02_pair_state.pkl"

candidates = [
    Path("output") / filename,
    Path("../output") / filename,
    Path("../../output") / filename,
]

pair_state_path = next((p.resolve() for p in candidates if p.exists()), None)

if pair_state_path is None:
    raise FileNotFoundError(
        f"{filename} not found. Checked:\n" +
        "\n".join(str(p.resolve()) for p in candidates)
    )

print("Found:", pair_state_path)

with open(pair_state_path, "rb") as f:
    pair_state = pickle.load(f)

print("Type:", type(pair_state))

if isinstance(pair_state, dict):
    for k, v in pair_state.items():
        size = len(v) if hasattr(v, "__len__") else "NA"
        print(f"{k}: {type(v)}, size={size}")

In [ ]:
pair_true_hits = pair_state["pair_true_hits"]

print("Threshold keys:", pair_true_hits.keys())

for threshold, value in pair_true_hits.items():
    print(
        f"\nThreshold <= {threshold}"
        f"\nType: {type(value)}"
        f"\nSize: {len(value) if hasattr(value, '__len__') else 'NA'}"
    )

    if isinstance(value, (set, list, tuple)):
        print("First 5:", list(value)[:5])
    elif isinstance(value, dict):
        print("First 5 keys:", list(value.keys())[:5])

In [ ]:
# True targets captured by the current blocker
# (exact name + name-token <=10 + address-token <=10)
print("Current blocker:", len(current_captured_true_targets))

# True targets captured by pair blocker <=2
pair2_true_targets = pair_true_hits[2]
print("Pair <=2:", len(pair2_true_targets))

# Union = actual current blocker + pair<=2 baseline
pair2_union_true_targets = (
    current_captured_true_targets
    | pair2_true_targets
)

print("Union:", len(pair2_union_true_targets))

In [ ]:
overlap = (
    current_captured_true_targets
    & pair2_true_targets
)

print("Overlap:", len(overlap))
print("Expected overlap: 502513")

print(
    "Recall:",
    len(pair2_union_true_targets) / pair_state["total_true_matches"]
)

In [ ]:
thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_matches = pair_state["total_true_matches"]

rows = []

for threshold in thresholds:

    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    new_matches = (
        target_route_targets
        - pair2_union_true_targets
    )

    union_targets = (
        pair2_union_true_targets
        | target_route_targets
    )

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "new_true_matches_added": len(new_matches),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_matches
    })

target_union_df = pd.DataFrame(rows)

target_union_df

In [74]:
from collections import Counter
from itertools import combinations

# --------------------------------------------------------------
# Target-side pair frequencies
# --------------------------------------------------------------

def build_target_pair_frequencies(
    source_path,
    source_name
):
    target_name_pair_frequency = Counter()
    target_address_pair_frequency = Counter()

    # Only count pair keys that actually exist on the S1 side.
    # This avoids generating/storing irrelevant pairs.
    relevant_name_pairs = set(name_pair_to_s1.keys())
    relevant_address_pairs = set(address_pair_to_s1.keys())

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            # --------------------------------------------------
            # NAME PAIRS
            # --------------------------------------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_name)
                if token in token_to_id
            ]

            if len(name_token_ids) >= 2:

                for pair in combinations(
                    sorted(set(name_token_ids)),
                    2
                ):

                    if pair in relevant_name_pairs:
                        target_name_pair_frequency[pair] += 1

            # --------------------------------------------------
            # ADDRESS PAIRS
            # --------------------------------------------------
            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_address)
                if token in token_to_id
            ]

            if len(address_token_ids) >= 2:

                for pair in combinations(
                    sorted(set(address_token_ids)),
                    2
                ):

                    if pair in relevant_address_pairs:
                        target_address_pair_frequency[pair] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    print(
        f"{source_name} target name pairs:",
        len(target_name_pair_frequency)
    )

    print(
        f"{source_name} target address pairs:",
        len(target_address_pair_frequency)
    )

    return (
        target_name_pair_frequency,
        target_address_pair_frequency
    )


# --------------------------------------------------------------
# Build S2 target-side pair frequencies
# --------------------------------------------------------------

(
    target_name_pair_frequency_2,
    target_address_pair_frequency_2
) = build_target_pair_frequencies(
    SOURCE2_PATH,
    "S2"
)


# --------------------------------------------------------------
# Build S3 target-side pair frequencies
# --------------------------------------------------------------

(
    target_name_pair_frequency_3,
    target_address_pair_frequency_3
) = build_target_pair_frequencies(
    SOURCE3_PATH,
    "S3"
)


print()
print(
    "S2 name pair frequencies:",
    len(target_name_pair_frequency_2)
)

print(
    "S2 address pair frequencies:",
    len(target_address_pair_frequency_2)
)

print(
    "S3 name pair frequencies:",
    len(target_name_pair_frequency_3)
)

print(
    "S3 address pair frequencies:",
    len(target_address_pair_frequency_3)
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S2 target name pairs: 184199
S2 target address pairs: 1021824
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed
S3 target name pairs: 185021
S3 target address pairs: 983097

S2 name pair frequencies: 184199
S2 address pair frequencies: 1021824
S3 name pair frequencies: 185021
S3 address pair frequencies: 983097


In [75]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np

TARGET_PAIR_THRESHOLDS = [10, 50, 100, 500, 1000, 5000]
MAX_TARGET_THRESHOLD = max(TARGET_PAIR_THRESHOLDS)

# Robust source paths
def resolve_source_path(filename):
    candidates = [
        Path("data/train") / filename,
        Path("../data/train") / filename,
        Path("../../data/train") / filename,
    ]

    for p in candidates:
        if p.exists():
            return p.resolve()

    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n"
        + "\n".join(str(p.resolve()) for p in candidates)
    )


SOURCE2_PATH = resolve_source_path("train_source2.tsv")
SOURCE3_PATH = resolve_source_path("train_source3.tsv")

print("S2:", SOURCE2_PATH)
print("S3:", SOURCE3_PATH)


def evaluate_target_pair_candidate_volume(
    source_path,
    source_name,
    target_name_pair_frequency,
    target_address_pair_frequency
):

    totals = {
        t: 0
        for t in TARGET_PAIR_THRESHOLDS
    }

    row_counts_sample = {
        t: []
        for t in TARGET_PAIR_THRESHOLDS
    }

    total_rows = 0
    zero_candidate_rows = 0

    max_candidates = {
        t: 0
        for t in TARGET_PAIR_THRESHOLDS
    }

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            # --------------------------------------------------
            # For this target row:
            # S1 index -> minimum target-side pair frequency
            # --------------------------------------------------
            min_freq_by_s1 = {}

            # --------------------------------------------------
            # NAME PAIRS
            # --------------------------------------------------
            row_name_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_name)
                if token in token_to_id
            ]

            if len(row_name_ids) >= 2:

                for pair in combinations(
                    sorted(row_name_ids),
                    2
                ):

                    freq = target_name_pair_frequency.get(pair)

                    if (
                        freq is None
                        or freq > MAX_TARGET_THRESHOLD
                    ):
                        continue

                    s1_postings = name_pair_to_s1.get(
                        pair,
                        []
                    )

                    for s1_idx in s1_postings:

                        previous = min_freq_by_s1.get(
                            s1_idx
                        )

                        if (
                            previous is None
                            or freq < previous
                        ):
                            min_freq_by_s1[s1_idx] = freq

            # --------------------------------------------------
            # ADDRESS PAIRS
            # --------------------------------------------------
            row_address_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_address)
                if token in token_to_id
            ]

            if len(row_address_ids) >= 2:

                for pair in combinations(
                    sorted(row_address_ids),
                    2
                ):

                    freq = target_address_pair_frequency.get(pair)

                    if (
                        freq is None
                        or freq > MAX_TARGET_THRESHOLD
                    ):
                        continue

                    s1_postings = address_pair_to_s1.get(
                        pair,
                        []
                    )

                    for s1_idx in s1_postings:

                        previous = min_freq_by_s1.get(
                            s1_idx
                        )

                        if (
                            previous is None
                            or freq < previous
                        ):
                            min_freq_by_s1[s1_idx] = freq

            # --------------------------------------------------
            # Count unique (target, S1) candidates
            # --------------------------------------------------
            counts = [
                0
                for _ in TARGET_PAIR_THRESHOLDS
            ]

            for freq in min_freq_by_s1.values():

                if freq <= 10:
                    counts[0] += 1

                elif freq <= 50:
                    counts[1] += 1

                elif freq <= 100:
                    counts[2] += 1

                elif freq <= 500:
                    counts[3] += 1

                elif freq <= 1000:
                    counts[4] += 1

                elif freq <= 5000:
                    counts[5] += 1

            # --------------------------------------------------
            # Convert bucket counts to cumulative counts
            # --------------------------------------------------
            for i in range(
                1,
                len(counts)
            ):
                counts[i] += counts[i - 1]

            if len(min_freq_by_s1) == 0:
                zero_candidate_rows += 1

            # --------------------------------------------------
            # Store threshold statistics
            # --------------------------------------------------
            for i, threshold in enumerate(
                TARGET_PAIR_THRESHOLDS
            ):

                c = counts[i]

                totals[threshold] += c

                if c > max_candidates[threshold]:
                    max_candidates[threshold] = c

                # Deterministic sample:
                # every 1000th target row
                if total_rows % 1000 == 0:
                    row_counts_sample[
                        threshold
                    ].append(c)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    # ----------------------------------------------------------
    # Build final result table
    # ----------------------------------------------------------
    results = []

    for threshold in TARGET_PAIR_THRESHOLDS:

        sample = np.array(
            row_counts_sample[threshold],
            dtype=np.int64
        )

        results.append({

            "source": source_name,

            "target_pair_frequency_threshold": (
                threshold
            ),

            "candidate_pairs": (
                totals[threshold]
            ),

            "avg_candidates_per_target": (
                totals[threshold] / total_rows
                if total_rows
                else 0
            ),

            "median_candidates_per_target_sample": (
                np.median(sample)
                if len(sample)
                else 0
            ),

            "p95_candidates_per_target_sample": (
                np.percentile(sample, 95)
                if len(sample)
                else 0
            ),

            "max_candidates_per_target": (
                max_candidates[threshold]
            ),

            "zero_candidate_target_pct": (
                100 * zero_candidate_rows / total_rows
                if total_rows
                else 0
            )
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------
# Run S2 and S3
# --------------------------------------------------------------

target_candidate_s2_df = (
    evaluate_target_pair_candidate_volume(
        SOURCE2_PATH,
        "S2",
        target_name_pair_frequency_2,
        target_address_pair_frequency_2
    )
)

target_candidate_s3_df = (
    evaluate_target_pair_candidate_volume(
        SOURCE3_PATH,
        "S3",
        target_name_pair_frequency_3,
        target_address_pair_frequency_3
    )
)

target_candidate_volume_df = pd.concat(
    [
        target_candidate_s2_df,
        target_candidate_s3_df
    ],
    ignore_index=True
)

target_candidate_volume_df

S2: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/data/train/train_source2.tsv
S3: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/data/train/train_source3.tsv
S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


,source,target_pair_frequency_threshold,candidate_pairs,avg_candidates_per_target,median_candidates_per_target_sample,p95_candidates_per_target_sample,max_candidates_per_target,zero_candidate_target_pct
0,S2,10,1829050,0.363295,0.0,2.00,358,9.926815
1,S2,50,10993680,2.183618,1.0,9.00,374,9.926815
2,S2,100,22949834,4.558408,2.0,17.00,528,9.926815
3,S2,500,112493668,22.344041,12.0,79.00,1036,9.926815
4,S2,1000,215792654,42.861790,21.0,154.35,1449,9.926815
5,S2,5000,916378652,182.015600,92.0,676.00,3970,9.926815
6,S3,10,2068039,0.391259,0.0,2.00,42,13.463951
7,S3,50,11927575,2.256616,1.0,9.00,165,13.463951
8,S3,100,25351063,4.796248,1.0,18.80,329,13.463951
9,S3,500,121065899,22.904842,7.0,86.80,1506,13.463951


In [81]:
from itertools import combinations
import pandas as pd

# --------------------------------------------------------------
# Target-side pair blocking recall on validation ground truth
# --------------------------------------------------------------

# Build:
# target_entity_id -> validation S1 entity_id
#
# validation_ground_truth structure:
# {
#     S1_ID: {S2_ID, S3_ID, ...}
# }

target_to_s1 = {}

for s1_id, matched_ids in validation_ground_truth.items():

    for target_id in matched_ids:

        if target_id in target_to_s1:
            raise ValueError(
                f"Target ID linked to multiple S1 entities: {target_id}"
            )

        target_to_s1[target_id] = s1_id


# Total number of true S1-target relationships
total_true_matches = sum(
    len(matched_ids)
    for matched_ids in validation_ground_truth.values()
)

print(
    "Validation S1 entities:",
    len(validation_ground_truth)
)

print(
    "Total true validation relationships:",
    total_true_matches
)


def evaluate_target_pair_recall(
    source_path,
    source_name,
    target_name_pair_frequency,
    target_address_pair_frequency
):

    captured = {
        threshold: set()
        for threshold in TARGET_PAIR_THRESHOLDS
    }

    total_true_in_source = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id

            true_s1_id = target_to_s1.get(
                target_id
            )

            # This target record is not a true match
            # to a validation S1 entity.
            if true_s1_id is None:
                continue

            total_true_in_source += 1

            # Get validation S1 index
            true_s1_idx = (
                validation_s1_id_to_index[true_s1_id]
            )

            min_freq = None

            # --------------------------------------------------
            # NAME PAIRS
            # --------------------------------------------------
            row_name_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_name)
                if token in token_to_id
            ]

            if len(row_name_ids) >= 2:

                for pair in combinations(
                    sorted(set(row_name_ids)),
                    2
                ):

                    freq = target_name_pair_frequency.get(
                        pair
                    )

                    if freq is None:
                        continue

                    if freq > MAX_TARGET_THRESHOLD:
                        continue

                    if true_s1_idx in name_pair_to_s1.get(
                        pair,
                        []
                    ):

                        if (
                            min_freq is None
                            or freq < min_freq
                        ):
                            min_freq = freq

            # --------------------------------------------------
            # ADDRESS PAIRS
            # --------------------------------------------------
            row_address_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_address)
                if token in token_to_id
            ]

            if len(row_address_ids) >= 2:

                for pair in combinations(
                    sorted(set(row_address_ids)),
                    2
                ):

                    freq = target_address_pair_frequency.get(
                        pair
                    )

                    if freq is None:
                        continue

                    if freq > MAX_TARGET_THRESHOLD:
                        continue

                    if true_s1_idx in address_pair_to_s1.get(
                        pair,
                        []
                    ):

                        if (
                            min_freq is None
                            or freq < min_freq
                        ):
                            min_freq = freq

            # --------------------------------------------------
            # Record captured relationship by threshold
            # --------------------------------------------------
            if min_freq is None:
                continue

            relationship = (
                true_s1_idx,
                target_id
            )

            for threshold in TARGET_PAIR_THRESHOLDS:

                if min_freq <= threshold:

                    captured[
                        threshold
                    ].add(
                        relationship
                    )

        if chunk_number % 10 == 0:

            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

    return captured, total_true_in_source


# --------------------------------------------------------------
# Validation S1 ID -> row index
# --------------------------------------------------------------

validation_s1_id_to_index = {
    row.s1_id: idx
    for idx, row in enumerate(
        validation_s1.itertuples(index=False)
    )
}

print(
    "Validation S1 ID index:",
    len(validation_s1_id_to_index)
)


# --------------------------------------------------------------
# Run S2
# --------------------------------------------------------------

target_recall_s2, total_true_s2 = (
    evaluate_target_pair_recall(
        SOURCE2_PATH,
        "S2",
        target_name_pair_frequency_2,
        target_address_pair_frequency_2
    )
)


# --------------------------------------------------------------
# Run S3
# --------------------------------------------------------------

target_recall_s3, total_true_s3 = (
    evaluate_target_pair_recall(
        SOURCE3_PATH,
        "S3",
        target_name_pair_frequency_3,
        target_address_pair_frequency_3
    )
)


# --------------------------------------------------------------
# Combine S2 + S3
# --------------------------------------------------------------

target_pair_recall_results = []

for threshold in TARGET_PAIR_THRESHOLDS:

    true_hits = (
        target_recall_s2[threshold]
        |
        target_recall_s3[threshold]
    )

    target_pair_recall_results.append({

        "target_pair_frequency_threshold": threshold,

        "true_matches_captured": len(
            true_hits
        ),

        "total_true_matches": (
            total_true_matches
        ),

        "candidate_recall": (
            len(true_hits)
            / total_true_matches
            if total_true_matches
            else 0
        )
    })


target_pair_recall_results_df = pd.DataFrame(
    target_pair_recall_results
)

target_pair_recall_results_df

Validation S1 entities: 220682
Total true validation relationships: 764045
Validation S1 ID index: 220682
S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


,target_pair_frequency_threshold,true_matches_captured,total_true_matches,candidate_recall
0,10,572862,764045,0.749775
1,50,645433,764045,0.844758
2,100,665036,764045,0.870415
3,500,692400,764045,0.906229
4,1000,699958,764045,0.916121
5,5000,714190,764045,0.934749


In [ ]:
target_candidate_volume_summary = (
    target_candidate_volume_df
    .groupby("target_pair_frequency_threshold", as_index=False)
    .agg({
        "candidate_pairs": "sum",
        "avg_candidates_per_target": "mean",
        "median_candidates_per_target_sample": "mean",
        "p95_candidates_per_target_sample": "mean",
        "max_candidates_per_target": "max"
    })
)

target_candidate_volume_summary

In [ ]:
heavy_token_comparison = []

for token in sorted(
    heavy_name_tokens
):
    heavy_token_comparison.append({
        "field": "name",
        "token": token,
        "validation_s1_frequency": (
            name_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_name_frequency.get(
                token,
                0
            )
        )
    })

for token in sorted(
    heavy_address_tokens
):
    heavy_token_comparison.append({
        "field": "address",
        "token": token,
        "validation_s1_frequency": (
            address_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_address_frequency.get(
                token,
                0
            )
        )
    })

heavy_token_comparison_df = pd.DataFrame(
    heavy_token_comparison
)

heavy_token_comparison_df = (
    heavy_token_comparison_df
    .sort_values(
        "s2_s3_target_frequency",
        ascending=False
    )
)

heavy_token_comparison_df.head(100)

,field,token,validation_s1_frequency,s2_s3_target_frequency
41,address,road,45852,1288416
44,address,street,31080,694453
21,address,drive,22288,457748
0,name,center,3455,414505
16,address,avenue,19106,405843
48,address,tx,13179,307571
27,address,lane,10699,243506
35,address,ny,10239,237129
49,address,unit,19020,219006
37,address,park,5159,201478


In [ ]:
# Target-side true targets at threshold 500
target_500_true_targets = {
    target_id
    for target_id, freq in true_target_min_pair_freq.items()
    if freq <= 500
}

# Full blocker true-target coverage
blocker_500_true_targets = (
    pair2_union_true_targets
    | target_500_true_targets
)

total_true_matches = pair_state["total_true_matches"]

print("Baseline pair<=2 union:", len(pair2_union_true_targets))
print("Target route <=500:", len(target_500_true_targets))
print("Final blocker <=500:", len(blocker_500_true_targets))

print(
    "Recall:",
    len(blocker_500_true_targets) / total_true_matches
)

print(
    "Remaining missed:",
    total_true_matches - len(blocker_500_true_targets)
)


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(OUTPUT_DIR)

blocking_artifact_path = OUTPUT_DIR / "blocking_artifacts_v1.pkl"

with open(blocking_artifact_path, "wb") as f:
    pickle.dump(
        blocking_artifact,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", blocking_artifact_path)

In [ ]:
# Complete blocking artifact for downstream notebooks

blocking_artifact.update({

    # Token vocabulary
    "token_to_id": token_to_id,

    # Exact normalized name
    "name_exact_map": dict(name_exact_map),

    # Single-token posting lists
    "name_token_to_s1": dict(name_token_to_s1),
    "address_token_to_s1": dict(address_token_to_s1),

    # IMPORTANT:
    # These frequencies are required to reproduce the
    # <=10 single-token blocking rule.
    "name_token_frequency": dict(name_token_frequency),
    "address_token_frequency": dict(address_token_frequency),

    # Pair blocking
    "name_pair_to_s1": dict(name_pair_to_s1),
    "address_pair_to_s1": dict(address_pair_to_s1),

    # Active S1-side pairs
    "active_name_pairs": active_name_pairs,
    "active_address_pairs": active_address_pairs,

    # Target-side pair frequencies
    "target_name_pair_frequency_2": target_name_pair_frequency_2,
    "target_address_pair_frequency_2": target_address_pair_frequency_2,

    # Final blocking thresholds
    "NAME_TOKEN_FREQUENCY_THRESHOLD": 10,
    "ADDRESS_TOKEN_FREQUENCY_THRESHOLD": 10,
    "PAIR_S1_THRESHOLD": 2,
    "TARGET_PAIR_THRESHOLD": 500,
})

blocking_artifact_path = OUTPUT_DIR / "blocking_artifacts_v1.pkl"

with open(
    blocking_artifact_path,
    "wb"
) as f:
    pickle.dump(
        blocking_artifact,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved complete blocking artifact:")
print(blocking_artifact_path)

print("\nKeys saved:")
for key in sorted(blocking_artifact.keys()):
    print(" -", key)